In [1]:
# Imports for Cross-Encoder models

import os, gc, time, copy, wandb, random, logging

import numpy as np
import pandas as pd
import polars as pl

from kaggle_secrets import UserSecretsClient

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold

import transformers
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from transformers import logging as hf_logging
from transformers.utils import logging as tf_logging

In [2]:
# Load datasets and encode label as `label_id`

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

LABEL_COL = "answer"

LABEL2ID = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

train_data = pl.read_csv(TRAIN_PATH).with_row_index("_idx")
test_data = pl.read_csv(TEST_PATH).with_row_index("_idx")
train_data = train_data.with_columns(
    pl.col(LABEL_COL).replace(LABEL2ID).cast(pl.Int8).alias("label_id")
)

In [3]:
# Configure data, models, device, and seeds for reproducibility

EPOCHS = 7
N_SPLITS = 5

ID_COL = "id"
QUESTION_COL = "prompt"
OPTION_COLS  = ["A", "B", "C", "D", "E"]

MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L12-v2"
# MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L6-v2"
# MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L2-v2"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()

SEED = 42

def set_seed(seed: int):
    
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)

# Create directory to store saved models

BEST_DIR = "/kaggle/working/best-cv-models"
os.makedirs(BEST_DIR, exist_ok=True)

# Configure logging levels to hide model-loading report

hf_logging.set_verbosity_error()

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

tf_logging.disable_progress_bar()

# Configure WandB login

wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: spandanjit2005 (spandanjit2005-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
# Define DataLoader seeding function, and Generator for reproducibility

def seed_worker(worker_id):
    
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)
val_generator = torch.Generator()
val_generator.manual_seed(SEED)

test_generator = torch.Generator()
test_generator.manual_seed(SEED)

In [5]:
# Procompute context for whole train and test data

def add_context(df):
    return df.with_columns(
        pl.struct([QUESTION_COL] + OPTION_COLS).map_elements(
            lambda row: "Question: " + str(row[QUESTION_COL]) + " Options: " + " | ".join(
                f"{c}) {str(row[c])}" for c in OPTION_COLS
            ),
            return_dtype=pl.Utf8
        ).alias("context")
    )

train_data = add_context(train_data)
test_data = add_context(test_data)

In [6]:
# Define Dataset class

class CEDataset(Dataset):
    
    def __init__(self, df, tokenizer, max_length=384):
        texts1 = []
        texts2 = []
        labels = []

        for row in df.iter_rows(named=True):
            context = row["context"]
            
            for c in OPTION_COLS:
                texts1.append(context)
                texts2.append(f"{c}) {str(row[c])}")
                labels.append(1.0 if c == row[LABEL_COL] else 0.0)

        enc = tokenizer(
            texts1,
            texts2,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )

        self.encodings = enc
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

In [7]:
# Define function to compute Mean Average Precision@3 (MAP@3)

def compute_map3(scores, df):
    scores = np.asarray(scores).reshape(len(df), 5)
    true = df["label_id"].to_numpy()

    top3 = np.argsort(-scores, axis=1)[:, :3]
    hit = top3 == true[:, None]
    ranks = np.where(hit.any(axis=1), hit.argmax(axis=1) + 1, 0)

    out = np.zeros_like(ranks, dtype=float)
    np.divide(1.0, ranks, out=out, where=ranks > 0)
    
    return float(out.mean())

In [8]:
# Define function to return test prediction scores given a cross-encoder model

def get_test_scores(model):
    tmp = test_data.with_columns(pl.lit("A").alias("answer"))
    
    test_loader = DataLoader(
        CEDataset(tmp, tokenizer),
        batch_size=32,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=test_generator
    )
    
    model.eval()
    scores_all = []

    with torch.no_grad():
        for batch in test_loader:
            labels = batch["labels"].to(device, non_blocking=True)
            inputs = {k: v.to(device, non_blocking=True) for k, v in batch.items() if k != "labels"}

            logits = model(**inputs).logits.squeeze(-1)
            scores_all.extend(logits.detach().cpu().numpy().tolist())

    return np.array(scores_all).reshape(len(test_data), 5)

In [9]:
# Define function to compute Out-of-fold scores

def compute_oof_scores(model, loader, device):
    model.eval()
    scores = []

    with torch.no_grad():
        for batch in loader:
            inputs = {
                k: v.to(device, non_blocking=True)
                for k, v in batch.items()
                if k != "labels"
            }
            logits = model(**inputs).logits.squeeze(-1)
            scores.extend(logits.detach().cpu().numpy().tolist())

    return np.array(scores)

In [10]:
# Configure CV constructor and Tokenizer

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

oof_scores = np.zeros((len(train_data), 5))
fold_best_paths = []

# Configure WandB logging

run = wandb.init(
    entity="24f2005537-dl-genai-project",
    project="dl-genai-project",
    config={
        "base_model": MODEL_NAME,
        "architecture": "BERT",
        "total_parameters": "~33.4M",
        "transformer_layers": 12,
        "hidden_size": 384,
        "max_sequence_length": "512_tokens",
        "total_splits": N_SPLITS,
        "epochs_p_split": EPOCHS,
        "optimizer": "AdamW",
        "lr": 2e-5,
        "loss": "CrossEntropyLoss",
        "scheduler": None
    },
)

# Start Training Loop

for fold, (train_idx, val_idx) in enumerate(kf.split(train_data)):
    fold_start = time.perf_counter()
    print(f"\nFOLD {fold+1}/{N_SPLITS}...")

    train_fold = train_data[train_idx]
    val_fold = train_data[val_idx]

    # Configure DataLoaders for current fold
    
    train_loader = DataLoader(
        CEDataset(train_fold, tokenizer),
        batch_size=32,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=train_generator
    )

    val_loader = DataLoader(
        CEDataset(val_fold, tokenizer),
        batch_size=32,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=val_generator
    )

    ce = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
    
    if torch.cuda.device_count() > 1:
        ce = nn.DataParallel(ce, device_ids=[0, 1])
    ce = ce.to(device)

    optimizer = torch.optim.AdamW(
        ce.module.parameters() if isinstance(ce, nn.DataParallel) else ce.parameters(),
        lr=2e-5,
        weight_decay=0.01
    )
    loss_fn = nn.CrossEntropyLoss()

    best_val_map3 = -1.0
    best_path = os.path.join(BEST_DIR, f"f_{fold+1}_best.pt")
    fold_best_paths.append(best_path)

    # Srart training for {EPOCHS} in current fold
    
    for epoch in range(EPOCHS):
        epoch_start = time.perf_counter()

        ce.train()
        train_loss_sum = 0.0
        train_logits_all = []
        train_labels_all = []

        for batch in train_loader:
            labels = batch["labels"].to(device, non_blocking=True)
            inputs = {k: v.to(device, non_blocking=True) for k, v in batch.items() if k != "labels"}

            optimizer.zero_grad()
            logits = ce(**inputs).logits.squeeze(-1)
            loss = loss_fn(logits, labels)

            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * labels.size(0)
            train_logits_all.extend(logits.detach().cpu().numpy().tolist())
            train_labels_all.extend(labels.detach().cpu().numpy().tolist())

        train_loss = train_loss_sum / len(train_fold)
        train_map3 = compute_map3(train_logits_all, train_fold)

        ce.eval()
        val_loss_sum = 0.0
        val_logits_all = []
        val_labels_all = []

        with torch.no_grad():
            for batch in val_loader:
                labels = batch["labels"].to(device, non_blocking=True)
                inputs = {k: v.to(device, non_blocking=True) for k, v in batch.items() if k != "labels"}

                logits = ce(**inputs).logits.squeeze(-1)
                loss = loss_fn(logits, labels)

                val_loss_sum += loss.item() * labels.size(0)
                val_logits_all.extend(logits.detach().cpu().numpy().tolist())
                val_labels_all.extend(labels.detach().cpu().numpy().tolist())

        val_loss = val_loss_sum / len(val_fold)
        val_map3 = compute_map3(val_logits_all, val_fold)

        epoch_time = time.perf_counter() - epoch_start

        run.log({
            "fold": fold+1,
            "epoch": epoch+1,
            "epoch_secs": round(epoch_time, 4),
            "train_loss": train_loss,
            "train_map3": train_map3,
            "val_loss": val_loss,
            "val_map3": val_map3
        })
        print(
            f"Fold: {fold+1}/{N_SPLITS} "
            f"Epoch: {epoch+1}/{EPOCHS} "
            f"Time to complete: {epoch_time:.2f}s\n"
            f"\tTrain Loss: {train_loss:.6f} Train MAP@3: {train_map3:.6f}\n"
            f"\tVal Loss: {val_loss:.6f} Val MAP@3: {val_map3:.6f}"
        )

        if val_map3 > best_val_map3:
            best_val_map3 = val_map3
            state = ce.module.state_dict() if isinstance(ce, nn.DataParallel) else ce.state_dict()
            torch.save(state, best_path)

    best_ce = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
    state = torch.load(best_path, map_location="cpu")
    best_ce.load_state_dict(state)
    
    if torch.cuda.device_count() > 1:
        best_ce = nn.DataParallel(best_ce, device_ids=[0, 1])
    
    best_ce = best_ce.to(device)

    best_val_logits = compute_oof_scores(best_ce, val_loader, device)
    oof_scores[val_idx] = best_val_logits.reshape(len(val_fold), 5)
    
    fold_time = time.perf_counter() - fold_start

    run.log({
        "fold": fold+1,
        "time_min": int(fold_time//60),
        "time_sec": round(fold_time%60, 2),
        "best_val_map3": best_val_map3
    })
    print(f"\nFold {fold+1} completed in {int(fold_time//60)} min {fold_time%60:.2f}s"
          f" | Best Val MAP@3: {best_val_map3:.6f}")

    del ce, best_ce, optimizer
    torch.cuda.empty_cache()
    gc.collect()

oof_map3 = compute_map3(oof_scores.flatten(), train_data)

run.log({
    "oof_map": oof_map3
})
print(f"\nOut-of-fold MAP@3 score: {oof_map3:.8f}")

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260601_112323-ucqvaoo8
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run olive-moon-1
wandb: ⭐️ View project at https://wandb.ai/24f2005537-dl-genai-project/dl-genai-project
wandb: 🚀 View run at https://wandb.ai/24f2005537-dl-genai-project/dl-genai-project/runs/ucqvaoo8



FOLD 1/5...
Fold: 1/5 Epoch: 1/7 Time to complete: 114.39s
	Train Loss: 116.492299 Train MAP@3: 0.367604
	Val Loss: 106.820994 Val MAP@3: 0.589583
Fold: 1/5 Epoch: 2/7 Time to complete: 120.64s
	Train Loss: 104.628708 Train MAP@3: 0.375417
	Val Loss: 92.102655 Val MAP@3: 0.781250
Fold: 1/5 Epoch: 3/7 Time to complete: 120.80s
	Train Loss: 90.362487 Train MAP@3: 0.375937
	Val Loss: 78.229172 Val MAP@3: 0.920417
Fold: 1/5 Epoch: 4/7 Time to complete: 120.66s
	Train Loss: 78.243629 Train MAP@3: 0.361354
	Val Loss: 68.446573 Val MAP@3: 0.983750
Fold: 1/5 Epoch: 5/7 Time to complete: 120.59s
	Train Loss: 71.565791 Train MAP@3: 0.351667
	Val Loss: 65.479736 Val MAP@3: 0.992500
Fold: 1/5 Epoch: 6/7 Time to complete: 121.15s
	Train Loss: 68.647577 Train MAP@3: 0.357396
	Val Loss: 63.812411 Val MAP@3: 0.990000
Fold: 1/5 Epoch: 7/7 Time to complete: 120.25s
	Train Loss: 65.952762 Train MAP@3: 0.363438
	Val Loss: 61.000020 Val MAP@3: 0.998750

Fold 1 completed in 14 min 21.24s | Best Val MAP@3: 

In [11]:
test_scores = np.zeros((len(test_data), 5))

for fold, best_path in enumerate(fold_best_paths):
    ce = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
    state = torch.load(best_path, map_location="cpu")
    ce.load_state_dict(state)

    if torch.cuda.device_count() > 1:
        ce = nn.DataParallel(ce, device_ids=[0, 1])
    ce = ce.to(device)

    test_scores += get_test_scores(ce) / N_SPLITS

    del ce
    torch.cuda.empty_cache()
    gc.collect()
    
top3_idx = np.argsort(-test_scores, axis=1)[:, :3]
pred_strings = [" ".join(OPTION_COLS[i] for i in row) for row in top3_idx]

submission = pl.DataFrame({"ID": test_data[ID_COL], "Prediction": pred_strings})
submission.write_csv("submission.csv")
print(submission.sample(5))

shape: (5, 2)
┌─────┬────────────┐
│ ID  ┆ Prediction │
│ --- ┆ ---        │
│ i64 ┆ str        │
╞═════╪════════════╡
│ 83  ┆ D B A      │
│ 152 ┆ B D C      │
│ 200 ┆ C A D      │
│ 61  ┆ C D A      │
│ 354 ┆ E B A      │
└─────┴────────────┘
